In [1]:
!wget --user-agent "Mozilla" "https://arxiv.org/pdf/2312.04511.pdf" -O "llm_compiler.pdf"
!wget --user-agent "Mozilla" "https://arxiv.org/pdf/2312.06648.pdf" -O "dense_x_retrieval.pdf"

--2025-09-30 14:27:32--  https://arxiv.org/pdf/2312.04511.pdf
Resolving arxiv.org (arxiv.org)... 151.101.195.42, 151.101.131.42, 151.101.3.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.195.42|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /pdf/2312.04511 [following]
--2025-09-30 14:27:37--  https://arxiv.org/pdf/2312.04511
Reusing existing connection to arxiv.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 1020527 (997K) [application/pdf]
Saving to: ‘llm_compiler.pdf’

llm_compiler.pdf    100%[===================>] 996.61K  5.37MB/s    in 0.2s    

2025-09-30 14:27:38 (5.37 MB/s) - ‘llm_compiler.pdf’ saved [1020527/1020527]

--2025-09-30 14:27:38--  https://arxiv.org/pdf/2312.06648.pdf
Resolving arxiv.org (arxiv.org)... 151.101.195.42, 151.101.67.42, 151.101.3.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.195.42|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /pdf/2312.066

In [3]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
## Load the data
reader = SimpleDirectoryReader(input_files=['data/dense_x_retrieval.pdf'])
documents_jerry = reader.load_data()

reader = SimpleDirectoryReader(input_files=['data/llm_compiler.pdf'])
documents_ravi = reader.load_data()

In [5]:
index = VectorStoreIndex.from_documents(documents=[])


In [6]:
from llama_index.core.ingestion import IngestionPipeline, IngestionCache
from llama_index.core.node_parser import SentenceSplitter

pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=512, chunk_overlap=20),
    ]
)

In [7]:
# For user Jerry
for document in documents_jerry:
    document.metadata['user'] = 'Jerry'

nodes = pipeline.run(documents=documents_jerry)
# Insert nodes into the index
index.insert_nodes(nodes)

# For user Ravi
for document in documents_ravi:
    document.metadata['user'] = 'Ravi'

nodes = pipeline.run(documents=documents_ravi)
# Insert nodes into the index
index.insert_nodes(nodes)

In [10]:
# For Jerry


from llama_index.core.vector_stores import ExactMatchFilter, MetadataFilters, FilterOperator
jerry_query_engine = index.as_query_engine(
    filters=MetadataFilters(
        filters=[
            ExactMatchFilter(
                key="user",
                value="Jerry",
            )
        ]
    ),
    similarity_top_k=3
)

# For Ravi
ravi_query_engine = index.as_query_engine(
    filters=MetadataFilters(
        filters=[
            ExactMatchFilter(
                key="user",
                value="Ravi",
            )
        ]
    ),
    similarity_top_k=3
)

In [13]:
# Jerry has Dense X Rerieval paper and should be able to answer following question.
response = jerry_query_engine.query(
    "what are propositions mentioned in the paper?"
)

In [14]:
response

Response(response='The paper discusses dense cross-retrieval and its applications in information retrieval systems.', source_nodes=[NodeWithScore(node=TextNode(id_='25e31aab-f5d4-4dd1-9ce3-a3fe065b1f52', embedding=None, metadata={'file_path': 'data/dense_x_retrieval.pdf', 'file_name': 'dense_x_retrieval.pdf', 'file_type': 'application/pdf', 'file_size': 935698, 'creation_date': '2025-09-30', 'last_modified_date': '2024-10-07', 'user': 'Jerry'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='6af9b1f8-5abd-4c4b-8cf6-5dd20d94cd34', node_type='4', metadata={'file_path': 'data/dense_x_retrieval.pdf', 'file_name': 'dense_x_retrieval.pdf', 'file_type': 'application/pdf', 'file_size': 935698, 'creation_date': '202

In [15]:
# Ravi has LLMCompiler paper
response = ravi_query_engine.query("what are steps involved in LLMCompiler?")


In [16]:
response

Response(response='The steps involved in LLMCompiler include reading the input source code, lexical analysis to break the source code into tokens, syntax analysis to create a parse tree, semantic analysis to check for semantic correctness, code generation to produce target code, and optimization to improve the efficiency of the generated code.', source_nodes=[NodeWithScore(node=TextNode(id_='df71621d-5b24-4a4d-90d3-682754998c9d', embedding=None, metadata={'file_path': 'data/llm_compiler.pdf', 'file_name': 'llm_compiler.pdf', 'file_type': 'application/pdf', 'file_size': 1020527, 'creation_date': '2025-09-30', 'last_modified_date': '2024-06-06', 'user': 'Ravi'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id=